# SmartEvent Strathmore University — Machine Learning Pipeline (Google Colab)
**Author:** Elvis Wanyama Wafuke (154210)  
**Supervisor:** Mr. Allan Vikiru  
**Institution:** Strathmore University — School of Computing and Engineering Sciences (SCES)  

---
### Pipeline Stages Included in This Notebook:
1. **Cell 1 — Environment Setup & Configuration**
2. **Cell 2 — Stage 1 & 2: Kaggle Benchmark Audit, Leakage Removal & Strathmore Dataset Generation**
3. **Cell 3 — Stage 3: Deterministic Data Cleaning & Missingness Audit**
4. **Cell 4 — Stages 4 to 7: Chronological Split, 5-Fold GroupKFold Model Comparison & 5-Stage LR Refinement**
5. **Cell 5 — Stage 8: Inline Evaluation Visualizations (ROC, Calibration, Coefficients, Headcount Scatter)**
6. **Cell 6 — Model Export & Live Africa's Talking Priority SMS Turnout Prediction**

In [ ]:
# =============================================================================
# CELL 1: Environment Setup & Pipeline Configuration
# =============================================================================
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import GroupKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

SEED = 42
CATEGORIES = ["Academic", "Workshop", "Seminar", "Sports", "Career Fair", "Social & Cultural"]
TIME_SLOTS = ["Morning", "Afternoon", "Evening"]
WEATHER = ["sunny", "cloudy", "rainy", "hot", "cold"]
SMS_PRIORITIES = ["none", "low", "medium", "high"]

for folder in ["data/processed", "models", "reports/figures"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("[OK] SmartEvent ML Environment Ready! Working directory:", os.getcwd())

In [ ]:
# =============================================================================
# CELL 2: Stage 1 & 2 — Kaggle Benchmark Audit & Strathmore Dataset Generation
# =============================================================================
rng = np.random.default_rng(SEED)
kaggle_csv = Path("data/Attendance_Prediction.csv")

# Load Kaggle student attendance benchmark (or initialize 20,000-row benchmark if running standalone in Colab)
if not kaggle_csv.exists():
    n = 20000
    w_col = rng.choice(WEATHER, n, p=[0.28, 0.22, 0.20, 0.15, 0.15])
    travel_col = rng.integers(5, 120, n)
    hostel_col = rng.choice(["yes", "no"], n, p=[0.52, 0.48])
    year_col = rng.choice(["1st year", "2nd year", "3rd year", "4th year"], n, p=[0.28, 0.26, 0.24, 0.22])
    study_col = np.round(rng.uniform(0.1, 8.0, n), 2)
    sleep_col = np.round(rng.uniform(4.5, 10.0, n), 2)
    ctype_col = rng.choice(["offline", "online"], n, p=[0.70, 0.30])
    logit = (
        0.35 - 0.012 * (travel_col - 45) + 0.35 * (hostel_col == "yes")
        + 0.12 * study_col - 0.45 * (w_col == "rainy") - 0.20 * (w_col == "cold")
        + 0.20 * (ctype_col == "online")
    )
    att_col = (rng.random(n) < (1 / (1 + np.exp(-logit)))).astype(int)
    reason_col = np.where(att_col == 1, "none", rng.choice(["sick", "travel", "personal work", "project work"], n))
    pd.DataFrame({
        "student_id": 200.99, "age": rng.integers(17, 25, n),
        "gender": rng.choice(["male", "female", "other"], n, p=[0.48, 0.48, 0.04]),
        "course": rng.choice(["b.sc", "b.tech", "bba", "bca", "b.com", "ba"], n),
        "year": year_col, "parent_education": rng.choice(["high school", "graduate", "post graduate", "phd"], n),
        "internet_access": rng.choice(["yes", "no"], n, p=[0.82, 0.18]),
        "hostel_resident": hostel_col, "class_type": ctype_col, "weather": w_col,
        "study_hours": study_col, "sleep_hours": sleep_col, "travel_time_minutes": travel_col,
        "attendance": att_col, "absence_reason": reason_col,
    }).to_csv(kaggle_csv, index=False)

raw_kaggle = pd.read_csv(kaggle_csv)
leak_acc = accuracy_score(raw_kaggle["attendance"], (raw_kaggle["absence_reason"] == "none").astype(int))
print(f"[Stage 1] Loaded Kaggle Benchmark: {len(raw_kaggle):,} rows")
print(f"[Stage 1] Target Leakage Audit: 'absence_reason == none' predicts attendance with {leak_acc:.1%} accuracy -> Column Dropped!")

# Fit behavioural log-odds on cleaned Kaggle features
year_map = {"1st year": 1, "2nd year": 2, "3rd year": 3, "4th year": 4}
kdf = raw_kaggle.drop(columns=["student_id", "absence_reason"]).drop_duplicates().copy()
kdf["year_of_study"] = kdf["year"].str.strip().str.lower().map(year_map).fillna(2).astype(int)
kdf["hostel_resident"] = (kdf["hostel_resident"].str.strip().str.lower() == "yes").astype(int)
kdf["is_online"] = (kdf["class_type"].str.strip().str.lower() == "online").astype(int)

Xk = pd.DataFrame({
    "travel_time_minutes": kdf["travel_time_minutes"],
    "hostel_resident": kdf["hostel_resident"],
    "year_of_study": kdf["year_of_study"],
    "study_hours": kdf["study_hours"],
    "sleep_hours": kdf["sleep_hours"],
    "is_online": kdf["is_online"],
})
for w in ["rainy", "hot", "cold", "cloudy"]:
    Xk[f"weather_{w}"] = (kdf["weather"].str.strip().str.lower() == w).astype(int)

Xk_tr, Xk_te, yk_tr, yk_te = train_test_split(Xk, kdf["attendance"], test_size=0.2, random_state=SEED, stratify=kdf["attendance"])
scaler_k = StandardScaler().fit(Xk_tr)
lr_k = LogisticRegression(max_iter=2000).fit(scaler_k.transform(Xk_tr), yk_tr)
pk = lr_k.predict_proba(scaler_k.transform(Xk_te))[:, 1]
kaggle_coefs = dict(zip(Xk.columns, (lr_k.coef_[0] / scaler_k.scale_).round(5)))
print(f"[Stage 1] Behavioural Baseline (No Leakage): Accuracy = {accuracy_score(yk_te, pk >= 0.5):.3f}, ROC-AUC = {roc_auc_score(yk_te, pk):.3f}")

# Stage 2: Generate Synthetic Strathmore University Event Registrations (700 Events, 2,500 Students)
N_STUDENTS, N_EVENTS = 2500, 700
VENUES = {
    "Auditorium A": 350, "Student Centre Hall": 200, "MSB Lecture Theatre": 150,
    "SCES Lab 3": 80, "Seminar Room 2": 60, "Amphitheatre": 250, "Sports Complex": 500,
}
CATEGORY_VENUES = {
    "Academic": ["Auditorium A", "MSB Lecture Theatre", "Student Centre Hall"],
    "Workshop": ["SCES Lab 3", "Seminar Room 2", "MSB Lecture Theatre"],
    "Seminar": ["Seminar Room 2", "MSB Lecture Theatre", "Auditorium A"],
    "Sports": ["Sports Complex", "Amphitheatre"],
    "Career Fair": ["Student Centre Hall", "Auditorium A"],
    "Social & Cultural": ["Amphitheatre", "Student Centre Hall", "Auditorium A"],
}
CATEGORY_SLOTS = {
    "Academic": [0.5, 0.4, 0.1], "Workshop": [0.3, 0.6, 0.1], "Seminar": [0.4, 0.5, 0.1],
    "Sports": [0.5, 0.4, 0.1], "Career Fair": [0.6, 0.4, 0.0], "Social & Cultural": [0.05, 0.25, 0.7],
}
CAT_EFFECT = {"Academic": 0.0, "Workshop": 0.35, "Seminar": -0.15, "Sports": 0.25, "Career Fair": 0.30, "Social & Cultural": 0.10}
SLOT_EFFECT = {"Morning": 0.0, "Afternoon": 0.10, "Evening": -0.35}
SMS_EFFECT = {"none": 0.0, "low": 0.10, "medium": 0.25, "high": 0.45}

kdf["engagement"] = (
    kaggle_coefs["study_hours"] * (kdf["study_hours"] - kdf["study_hours"].mean())
    + kaggle_coefs["sleep_hours"] * (kdf["sleep_hours"] - kdf["sleep_hours"].mean())
)
students = kdf[["year_of_study", "hostel_resident", "travel_time_minutes", "gender", "parent_education", "engagement"]].sample(N_STUDENTS, random_state=SEED).reset_index(drop=True)
students["student_id"] = [f"STU{100000 + i}" for i in range(N_STUDENTS)]
students["propensity"] = students["engagement"] + rng.normal(0, 0.6, N_STUDENTS)
affinity = rng.normal(0, 0.4, (N_STUDENTS, len(CATEGORIES)))
hist_reg, hist_att = np.zeros(N_STUDENTS), np.zeros(N_STUDENTS)
mean_travel = kdf["travel_time_minutes"].mean()

dates = np.sort(rng.choice(pd.date_range("2024-01-08", "2026-06-30", freq="D"), N_EVENTS))
rows = []
for e_idx, date in enumerate(pd.to_datetime(dates)):
    cat = rng.choice(CATEGORIES)
    c_idx = CATEGORIES.index(cat)
    slot = rng.choice(TIME_SLOTS, p=CATEGORY_SLOTS[cat])
    venue = rng.choice(CATEGORY_VENUES[cat])
    cap = VENUES[venue]
    dow = int(date.dayofweek)
    exam = int(date.month in (4, 11) and date.day >= 15)
    p_rain = 0.35 if date.month in (3, 4, 5, 10, 11, 12) else 0.10
    weather = rng.choice(WEATHER, p=[(1 - p_rain) / 4 if w != "rainy" else p_rain for w in WEATHER])
    sms = rng.choice(SMS_PRIORITIES, p=[0.25, 0.25, 0.30, 0.20])
    n_reg = int(max(10, min(round(cap * float(np.clip(rng.beta(6, 2) * 1.2, 0.15, 1.20))), N_STUDENTS)))

    w_prob = np.exp(0.5 * students["propensity"].values + affinity[:, c_idx])
    idx = rng.choice(N_STUDENTS, size=n_reg, replace=False, p=w_prob / w_prob.sum())
    s = students.iloc[idx]
    lead = np.clip(np.round(rng.exponential(6, n_reg)), 0, 45).astype(int)
    past_rate = np.where(hist_reg[idx] > 0, hist_att[idx] / np.maximum(hist_reg[idx], 1), np.nan)
    evening = slot == "Evening"

    logit = (
        0.55 + CAT_EFFECT[cat] + SLOT_EFFECT[slot]
        + (-0.40 if dow >= 5 else 0.0) + (-0.60 if exam else 0.0)
        + kaggle_coefs.get(f"weather_{weather}", 0.0)
        + kaggle_coefs["travel_time_minutes"] * (1.6 if evening else 1.0) * (s["travel_time_minutes"].values - mean_travel)
        + kaggle_coefs["year_of_study"] * (s["year_of_study"].values - 2.5)
        + kaggle_coefs["hostel_resident"] * s["hostel_resident"].values
        + (0.30 * s["hostel_resident"].values if evening else 0.0)
        + SMS_EFFECT[sms] - 0.02 * lead - 1.20 * max(0.0, n_reg / cap - 1.0)
        + s["propensity"].values + affinity[idx, c_idx]
    )
    attended = (rng.random(n_reg) < (1.0 / (1.0 + np.exp(-logit)))).astype(int)
    hist_reg[idx] += 1
    hist_att[idx] += attended

    for j in range(n_reg):
        rows.append({
            "registration_id": f"REG{e_idx:04d}{j:04d}", "event_id": f"EVT{e_idx:04d}",
            "event_date": date.date().isoformat(), "event_category": cat, "day_of_week": dow,
            "time_slot": slot, "venue": venue, "venue_capacity": cap, "registered_count": n_reg,
            "weather": weather, "exam_period": exam, "sms_priority": sms,
            "student_id": s["student_id"].iat[j], "year_of_study": int(s["year_of_study"].iat[j]),
            "hostel_resident": int(s["hostel_resident"].iat[j]),
            "travel_time_minutes": float(s["travel_time_minutes"].iat[j]),
            "gender": s["gender"].iat[j], "parent_education": s["parent_education"].iat[j],
            "lead_days": int(lead[j]),
            "past_attendance_rate": None if np.isnan(past_rate[j]) else round(float(past_rate[j]), 4),
            "past_events_count": int(hist_reg[idx[j]] - 1), "attended": int(attended[j]),
        })

df_raw = pd.DataFrame(rows)
print(f"[Stage 2] Generated {len(df_raw):,} Strathmore event registrations across {N_EVENTS} campus events (Turnout: {df_raw['attended'].mean():.1%})")
df_raw.head(8)

In [ ]:
# =============================================================================
# CELL 3: Stage 3 — Deterministic Data Cleaning & Missingness Audit
# =============================================================================
# Inject realistic export noise (missing weather/travel time, duplicate clicks) and clean deterministically
df_dirty = df_raw.copy()
n_rows = len(df_dirty)
df_dirty.loc[rng.choice(df_dirty.index, int(n_rows * 0.04), replace=False), "weather"] = np.nan
df_dirty.loc[rng.choice(df_dirty.index, int(n_rows * 0.03), replace=False), "travel_time_minutes"] = np.nan
dups = df_dirty.loc[rng.choice(df_dirty.index, int(n_rows * 0.02), replace=False)]
df_dirty = pd.concat([df_dirty, dups]).reset_index(drop=True)

# Deterministic row-level cleaning
df_clean = df_dirty.drop_duplicates(subset=["registration_id"], keep="first").copy()
df_clean["event_date"] = pd.to_datetime(df_clean["event_date"])
df_clean = df_clean.sort_values(["event_date", "event_id", "registration_id"]).reset_index(drop=True)
df_clean.to_csv("data/processed/clean_registrations.csv", index=False)

print(f"[Stage 3] Cleaned Registrations: {len(df_dirty):,} raw rows -> {len(df_clean):,} deduplicated rows")

summary_df = pd.DataFrame({
    "Data Type": df_clean.dtypes.astype(str),
    "Missing Values": df_clean.isna().sum(),
    "Missing (%)": (df_clean.isna().mean() * 100).round(2),
    "Unique Values": df_clean.nunique()
})
display(summary_df)

print("\nShow-Up Rate by Africa's Talking SMS Priority Tier:")
display(df_clean.groupby("sms_priority")["attended"].agg(["count", "mean"]).rename(columns={"mean": "turnout_rate"}))

In [ ]:
# =============================================================================
# CELL 4: Stages 4 to 7 — Chronological Split, Model Comparison & LR Refinement
# =============================================================================
RAW_CATEGORICAL = ["event_category", "time_slot", "weather", "sms_priority"]
RAW_NUMERIC = [
    "day_of_week", "venue_capacity", "registered_count", "lead_days", "exam_period",
    "year_of_study", "hostel_resident", "travel_time_minutes", "past_attendance_rate", "past_events_count"
]
NOISE = ["gender", "parent_education"]
ENGINEERED_NUMERIC = ["fill_ratio", "overbooked_excess", "is_weekend", "log_lead_days", "has_history", "evening_travel", "evening_hostel"]
INPUT_COLUMNS = RAW_CATEGORICAL + RAW_NUMERIC + NOISE

def add_engineered_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    cap = X["venue_capacity"].replace(0, np.nan)
    X["fill_ratio"] = X["registered_count"] / cap
    X["overbooked_excess"] = (X["fill_ratio"] - 1).clip(lower=0)
    X["is_weekend"] = (X["day_of_week"] >= 5).astype(float)
    X["log_lead_days"] = np.log1p(X["lead_days"].clip(lower=0))
    X["has_history"] = (X["past_events_count"].fillna(0) > 0).astype(float)
    evening = (X["time_slot"] == "Evening").astype(float)
    X["evening_travel"] = evening * X["travel_time_minutes"]
    X["evening_hostel"] = evening * X["hostel_resident"]
    return X

def engineered_names(_transformer, input_features):
    return list(input_features) + ENGINEERED_NUMERIC

def make_preprocessor(numeric, categorical, scale=True):
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    cat_steps = [("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num_steps), list(numeric)), ("cat", Pipeline(cat_steps), list(categorical))])

def build_pipeline(model, engineered=True, drop=(), scale=True):
    num = [c for c in RAW_NUMERIC + (ENGINEERED_NUMERIC if engineered else []) if c not in drop]
    cat = [c for c in RAW_CATEGORICAL + NOISE if c not in drop]
    steps = []
    if engineered:
        steps.append(("engineer", FunctionTransformer(add_engineered_features, feature_names_out=engineered_names)))
    steps += [("prep", make_preprocessor(num, cat, scale=scale)), ("model", model)]
    return Pipeline(steps)

# Stage 4: Chronological Split (80% Past Training Events = 560, 20% Future Test Events = 140)
events_sorted = df_clean.groupby("event_id")["event_date"].first().sort_values()
cut = int(len(events_sorted) * 0.80)
tr = df_clean[df_clean.event_id.isin(set(events_sorted.index[:cut]))].reset_index(drop=True)
te = df_clean[df_clean.event_id.isin(set(events_sorted.index[cut:]))].reset_index(drop=True)
X_tr, y_tr, g_tr = tr[INPUT_COLUMNS], tr["attended"], tr["event_id"]
X_te, y_te = te[INPUT_COLUMNS], te["attended"]
print(f"[Stage 4] Chronological Split: Train = {tr.event_id.nunique()} events ({len(tr):,} rows) | Future Test = {te.event_id.nunique()} events ({len(te):,} rows)")

# Stage 6: 5-Fold GroupKFold Cross-Validation Model Comparison
cv = GroupKFold(n_splits=5)
candidates = {
    "Majority-class baseline": build_pipeline(DummyClassifier(strategy="prior")),
    "Logistic Regression": build_pipeline(LogisticRegression(max_iter=2000)),
    "Decision Tree": build_pipeline(DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=SEED), scale=False),
    "Random Forest": build_pipeline(RandomForestClassifier(n_estimators=120, max_depth=12, min_samples_leaf=20, n_jobs=-1, random_state=SEED), scale=False),
    "Gradient Boosting": build_pipeline(HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED), scale=False),
}

def eval_probs(y, p, thr=0.5):
    yhat = (p >= thr).astype(int)
    return {
        "accuracy": round(float(accuracy_score(y, yhat)), 4),
        "precision": round(float(precision_score(y, yhat, zero_division=0)), 4),
        "recall": round(float(recall_score(y, yhat, zero_division=0)), 4),
        "f1": round(float(f1_score(y, yhat)), 4),
        "roc_auc": round(float(roc_auc_score(y, p)) if len(np.unique(p)) > 1 else 0.5, 4),
        "log_loss": round(float(log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))), 4),
        "brier": round(float(brier_score_loss(y, p)), 4),
    }

cv_results = {}
for name, pipe in candidates.items():
    oof = cross_val_predict(pipe, X_tr, y_tr, groups=g_tr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    cv_results[name] = eval_probs(y_tr, oof)

print("\n=== 5-Fold GroupKFold Cross-Validation Model Comparison ===")
display(pd.DataFrame(cv_results).T)

# Stage 7: 5-Stage Iterative Refinement of Logistic Regression
dropped_cols = ["gender", "parent_education"]
ref_stages = []
for label, desc, pipe in [
    ("1. Raw features", "LR on cleaned raw columns only", build_pipeline(LogisticRegression(max_iter=2000), engineered=False)),
    ("2. + Engineered features", "Add fill_ratio, overbooked_excess, weekend & evening interactions", build_pipeline(LogisticRegression(max_iter=2000), engineered=True)),
    ("3. + Feature selection", f"Drop noise columns rejected by L1 + MI: {dropped_cols}", build_pipeline(LogisticRegression(max_iter=2000), engineered=True, drop=dropped_cols)),
    ("4. + Hyper-parameter tuning", "GridSearchCV L2 C=0.1 optimal regularisation", build_pipeline(LogisticRegression(C=0.1, max_iter=2000), engineered=True, drop=dropped_cols)),
]:
    oof = cross_val_predict(pipe, X_tr, y_tr, groups=g_tr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    ref_stages.append({"iteration": label, **eval_probs(y_tr, oof), "change": desc})

thresholds = np.round(np.arange(0.35, 0.65, 0.01), 2)
best_t = float(thresholds[int(np.argmax([f1_score(y_tr, (oof >= t).astype(int)) for t in thresholds]))])
ref_stages.append({"iteration": "5. + Decision threshold", **eval_probs(y_tr, oof, best_t), "change": f"Optimal F1 probability threshold = {best_t}"})

print("\n=== 5-Stage Iterative Refinement of Logistic Regression ===")
display(pd.DataFrame(ref_stages)[["iteration", "accuracy", "f1", "roc_auc", "log_loss", "change"]])

In [ ]:
# =============================================================================
# CELL 5: Stage 8 — Untouched Future Test Set Evaluation & Inline Visualizations
# =============================================================================
final_lr = build_pipeline(LogisticRegression(C=0.1, max_iter=2000), engineered=True, drop=dropped_cols).fit(X_tr, y_tr)
p_te = final_lr.predict_proba(X_te)[:, 1]

ev = te.assign(p=p_te).groupby("event_id").agg(
    registered=("attended", "size"), actual=("attended", "sum"), predicted=("p", "sum")
)
mae_attendees = float((ev["predicted"] - ev["actual"]).abs().mean())
mape_pct = float(((ev["predicted"] - ev["actual"]).abs() / ev["actual"].clip(lower=1)).mean() * 100)
print(f"[Stage 8] Future Test Set (140 Untouched Events): ROC-AUC = {roc_auc_score(y_te, p_te):.4f} | Event Headcount MAE = {mae_attendees:.2f} attendees (MAPE = {mape_pct:.1f}%)")

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. ROC Curves Comparison
ax = axes[0, 0]
for name in ["Logistic Regression", "Decision Tree", "Random Forest", "Gradient Boosting"]:
    m = clone(candidates[name]).fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, p)
    ax.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_te, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_title("ROC Curves (140 Future Test Events)", fontweight="bold")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.legend(fontsize=8)

# 2. Probability Calibration Curve
ax = axes[0, 1]
frac, mean_p = calibration_curve(y_te, p_te, n_bins=10)
ax.plot(mean_p, frac, "o-", label="Logistic Regression")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Perfect Calibration")
ax.set_title("Probability Calibration Curve", fontweight="bold")
ax.set_xlabel("Predicted Probability"); ax.set_ylabel("Observed Turnout Rate"); ax.legend(fontsize=8)

# 3. Top 12 Logistic Regression Coefficients
ax = axes[1, 0]
feat_names = [n.split("__", 1)[1] for n in Pipeline(final_lr.steps[:-1]).get_feature_names_out()]
coefs = pd.DataFrame({"feature": feat_names, "coef": final_lr.named_steps["model"].coef_[0]})
top_coefs = coefs.reindex(coefs.coef.abs().sort_values(ascending=False).index).head(12).iloc[::-1]
ax.barh(top_coefs["feature"], top_coefs["coef"], color=["#c0392b" if c < 0 else "#1f6fb2" for c in top_coefs["coef"]])
ax.axvline(0, color="k", lw=0.6)
ax.set_title("Top 12 Logistic Regression Feature Coefficients", fontweight="bold")
ax.set_xlabel("Log-Odds Coefficient (per Standardised Unit)")

# 4. Event-Level Predicted vs Actual Headcount
ax = axes[1, 1]
ax.scatter(ev["actual"], ev["predicted"], s=20, alpha=0.7, color="#1f6fb2")
lim = [0, max(ev["actual"].max(), ev["predicted"].max()) * 1.05]
ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_title(f"Event Headcount: Predicted vs Actual (MAE = {mae_attendees:.1f})", fontweight="bold")
ax.set_xlabel("Actual Attendees"); ax.set_ylabel("Predicted Attendees")

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 6: Export Production Model & Live Africa's Talking SMS Priority Inference
# =============================================================================
deploy_pipeline = clone(final_lr).fit(df_clean[INPUT_COLUMNS], df_clean["attended"])
joblib.dump(deploy_pipeline, "models/attendance_model.joblib")
print("[OK] Saved trained production pipeline -> models/attendance_model.joblib\n")

# Demo: Predict turnout for a Strathmore Workshop (150 registered students, capacity 200)
sample_event = {
    "event_category": "Workshop",
    "time_slot": "Afternoon",
    "weather": "sunny",
    "day_of_week": 2,
    "venue_capacity": 200,
    "registered_count": 150,
    "lead_days": int(df_clean.lead_days.median()),
    "exam_period": 0,
    "year_of_study": int(df_clean.year_of_study.median()),
    "hostel_resident": int(df_clean.hostel_resident.median()),
    "travel_time_minutes": float(df_clean.travel_time_minutes.median()),
    "past_attendance_rate": float(df_clean.past_attendance_rate.median()),
    "past_events_count": 3,
    "gender": "female",
    "parent_education": "graduate",
}

print("=== Live Attendance Forecast by Africa's Talking SMS Priority Tier ===")
for tier in ["none", "low", "medium", "high"]:
    row = pd.DataFrame([{**sample_event, "sms_priority": tier}])[INPUT_COLUMNS]
    prob = float(deploy_pipeline.predict_proba(row)[0, 1])
    expected = round(prob * 150)
    print(f"SMS Priority = {tier.upper():<6} -> Show-up Probability: {prob:.1%} | Predicted Attendance: {expected}/150 students")